# Track B Cluster Notebook

This notebook runs the Track B pipeline on a cluster:
- Stage 1: MIND-SSC descriptor retrieval
- Stage 2: label-free re-ranking on the shortlist

Fill in `DATA_ROOT` and the six manifest paths before executing.

## Environment

Expected packages:
- `numpy`
- `torch`
- `nibabel`

If needed on the cluster, install them in the first code cell.

In [ ]:
# Optional on a fresh cluster environment:
# %pip install -e .
# %pip install numpy torch nibabel

from __future__ import annotations

import importlib
from pathlib import Path
from pprint import pprint

import track_b.io as track_b_io
import track_b.mind as track_b_mind
import track_b.rerank as track_b_rerank
import track_b.metrics as track_b_metrics

importlib.reload(track_b_io)
importlib.reload(track_b_mind)
importlib.reload(track_b_rerank)

from track_b.io import read_manifest, write_ranking_csv
from track_b.mind import DescriptorCache, mind_vector, similarity_matrix
from track_b.rerank import rerank_ranked_candidates
from track_b.metrics import score_prediction_file

DATA_ROOT = Path("/path/to/kaggle_dataset")
WORK_DIR = Path("./track_b_runs")
CACHE_DIR = WORK_DIR / "mind_cache"
STAGE1_DIR = WORK_DIR / "stage1"
STAGE2_DIR = WORK_DIR / "stage2"
WORK_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
STAGE1_DIR.mkdir(parents=True, exist_ok=True)
STAGE2_DIR.mkdir(parents=True, exist_ok=True)

POOL_SPECS = [
    {
        "name": "dataset1_val",
        "queries": DATA_ROOT / "dataset1" / "val_queries.csv",
        "gallery": DATA_ROOT / "dataset1" / "val_gallery.csv",
    },
    {
        "name": "dataset1_test",
        "queries": DATA_ROOT / "dataset1" / "test_queries.csv",
        "gallery": DATA_ROOT / "dataset1" / "test_gallery.csv",
    },
    {
        "name": "dataset2_val",
        "queries": DATA_ROOT / "dataset2" / "val_queries.csv",
        "gallery": DATA_ROOT / "dataset2" / "val_gallery.csv",
    },
    {
        "name": "dataset2_test",
        "queries": DATA_ROOT / "dataset2" / "test_queries.csv",
        "gallery": DATA_ROOT / "dataset2" / "test_gallery.csv",
    },
    {
        "name": "dataset3_val",
        "queries": DATA_ROOT / "dataset3" / "val_queries.csv",
        "gallery": DATA_ROOT / "dataset3" / "val_gallery.csv",
    },
    {
        "name": "dataset3_test",
        "queries": DATA_ROOT / "dataset3" / "test_queries.csv",
        "gallery": DATA_ROOT / "dataset3" / "test_gallery.csv",
    },
]

POOL_SPECS

## Stage 1: descriptor ranking

This computes a full gallery ranking for each pool and writes one CSV per pool.

In [ ]:
def rank_pool(pool_spec, *, max_side: int = 96):
    query_entries = read_manifest(pool_spec["queries"], root=DATA_ROOT)
    gallery_entries = read_manifest(pool_spec["gallery"], root=DATA_ROOT)
    cache = DescriptorCache(CACHE_DIR)

    gallery_vectors = []
    gallery_ids = []
    for entry in gallery_entries:
        gallery_vectors.append(cache.get(entry.image_path, max_side=max_side))
        gallery_ids.append(entry.item_id)

    rows = []
    for entry in query_entries:
        query_vector = cache.get(entry.image_path, max_side=max_side)
        scores = [float(query_vector @ gallery_vector) for gallery_vector in gallery_vectors]
        order = sorted(range(len(gallery_ids)), key=lambda idx: scores[idx], reverse=True)
        rows.append((entry.item_id, [gallery_ids[idx] for idx in order]))

    output_path = STAGE1_DIR / f"{pool_spec['name']}.csv"
    write_ranking_csv(rows, output_path)
    return output_path, rows


stage1_outputs = []
stage1_rows = {}
for pool_spec in POOL_SPECS:
    output_path, rows = rank_pool(pool_spec)
    stage1_outputs.append(output_path)
    stage1_rows[pool_spec["name"]] = rows

stage1_outputs

## Stage 2: re-rank top-k

This re-orders only the shortlist using label-free geometry/MIND scoring.

In [ ]:
def rerank_pool(pool_spec, *, top_k: int = 10, max_side: int = 96):
    query_entries = {entry.item_id: entry for entry in read_manifest(pool_spec["queries"], root=DATA_ROOT)}
    gallery_entries = {entry.item_id: entry for entry in read_manifest(pool_spec["gallery"], root=DATA_ROOT)}
    input_rows = stage1_rows[pool_spec["name"]]

    rows = []
    for query_id, ranked_ids in input_rows:
        query_entry = query_entries[query_id]
        candidates = [(target_id, gallery_entries[target_id].image_path) for target_id in ranked_ids[:top_k]]
        reranked = rerank_ranked_candidates(
            query_entry.image_path,
            candidates,
            top_k=top_k,
            max_side=max_side,
        )
        reranked_ids = [item.target_id for item in reranked]
        remaining = [target_id for target_id in ranked_ids if target_id not in set(reranked_ids)]
        rows.append((query_id, reranked_ids + remaining))

    output_path = STAGE2_DIR / f"{pool_spec['name']}.csv"
    write_ranking_csv(rows, output_path)
    return output_path, rows


stage2_outputs = []
stage2_rows = {}
for pool_spec in POOL_SPECS:
    output_path, rows = rerank_pool(pool_spec)
    stage2_outputs.append(output_path)
    stage2_rows[pool_spec["name"]] = rows

stage2_outputs

## Sanity checks

Check that every pool wrote a CSV and that each ranking is the full gallery length.

In [ ]:
for pool_spec in POOL_SPECS:
    name = pool_spec["name"]
    print(name, len(stage2_rows[name]))
    print("  stage1:", stage1_outputs[POOL_SPECS.index(pool_spec)])
    print("  stage2:", stage2_outputs[POOL_SPECS.index(pool_spec)])
    print("  first query ranking length:", len(stage2_rows[name][0][1]))


## Local evaluation split

This creates a deterministic train/validation split from `dataset1/train_pairs.csv` so the score uses matching query IDs and target IDs.

In [ ]:
import csv
import random

LOCAL_EVAL_DIR = WORK_DIR / "local_eval"
LOCAL_EVAL_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_SPLIT_SEED = 20260626
LOCAL_VAL_SIZE = 50

def load_pairs(path: Path):
    with Path(path).open("r", newline="") as handle:
        return list(csv.DictReader(handle))


def write_local_split(split_name: str, rows):
    queries_csv = LOCAL_EVAL_DIR / f"{split_name}_queries.csv"
    gallery_csv = LOCAL_EVAL_DIR / f"{split_name}_gallery.csv"
    labels_csv = LOCAL_EVAL_DIR / f"{split_name}_labels.csv"

    with queries_csv.open("w", newline="") as handle:
        writer = csv.writer(handle)
        writer.writerow(["query_id", "query_image", "query_modality", "dataset"])
        for row in rows:
            writer.writerow([row["query_id"], row["query_image"], row["query_modality"], row["dataset"]])

    with gallery_csv.open("w", newline="") as handle:
        writer = csv.writer(handle)
        writer.writerow(["target_id", "target_image", "target_modality", "dataset"])
        for row in rows:
            writer.writerow([row["target_id"], row["target_image"], row["target_modality"], row["dataset"]])

    with labels_csv.open("w", newline="") as handle:
        writer = csv.writer(handle)
        writer.writerow(["query_id", "target_id", "dataset"])
        for row in rows:
            writer.writerow([row["query_id"], row["target_id"], row["dataset"]])

    return queries_csv, gallery_csv, labels_csv


pairs = load_pairs(DATA_ROOT / "dataset1" / "train_pairs.csv")
indices = list(range(len(pairs)))
random.Random(LOCAL_SPLIT_SEED).shuffle(indices)
val_indices = set(indices[:LOCAL_VAL_SIZE])
local_val_rows = [pairs[index] for index in indices[:LOCAL_VAL_SIZE]]
local_train_rows = [pairs[index] for index in indices[LOCAL_VAL_SIZE:]]

local_train_queries, local_train_gallery, local_train_labels = write_local_split("train", local_train_rows)
local_val_queries, local_val_gallery, local_val_labels = write_local_split("val", local_val_rows)

print({
    "train": len(local_train_rows),
    "val": len(local_val_rows),
    "train_queries": local_train_queries,
    "val_queries": local_val_queries,
})


## Local predictions

Run Stage 1 and Stage 2 on the local split, then score the outputs.

In [ ]:
local_split_specs = [
    {"name": "local_train", "queries": local_train_queries, "gallery": local_train_gallery},
    {"name": "local_val", "queries": local_val_queries, "gallery": local_val_gallery},
]

for pool_spec in local_split_specs:
    stage1_path, rows = rank_pool(pool_spec)
    stage1_rows[pool_spec["name"]] = rows
    stage2_path, reranked_rows = rerank_pool(pool_spec)
    stage2_rows[pool_spec["name"]] = reranked_rows
    print(pool_spec["name"], stage1_path, stage2_path)


## Score predictions

This prints local MRR for the deterministic train/validation split using the in-memory rankings from the same split.

In [ ]:
def print_scores(name: str, label_rows, prediction_rows):
    from track_b.metrics import RetrievalLabel, mean_reciprocal_rank_by_dataset

    labels = [
        RetrievalLabel(query_id=row["query_id"], target_id=row["target_id"], dataset=row["dataset"])
        for row in label_rows
    ]
    rankings = {query_id: ranking for query_id, ranking in prediction_rows}
    scores = mean_reciprocal_rank_by_dataset(labels, rankings)
    print(f"{name}:", scores)
    return scores


train_scores = print_scores("train", local_train_rows, stage2_rows["local_train"])
val_scores = print_scores("val", local_val_rows, stage2_rows["local_val"])

train_scores, val_scores
